In [3]:
!pip install -U transformers accelerate peft bitsandbytes -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.

torchvision 0.13.1+cu116 requires torch==1.12.1, but you have torch 2.1.0 which is incompatible.

torchaudio 0.12.1+cu116 requires torch==1.12.1, but you have torch 2.1.0 which is incompatible.




In [4]:
!nvidia-smi

Sun Nov  5 13:39:54 2023       

+-----------------------------------------------------------------------------+

| NVIDIA-SMI 525.116.04   Driver Version: 525.116.04   CUDA Version: 12.0     |

|-------------------------------+----------------------+----------------------+

| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |

| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |

|                               |                      |               MIG M. |

|===============================+======================+======================|

|   0  NVIDIA RTX A4000    Off  | 00000000:00:05.0 Off |                  Off |

| 41%   30C    P8    12W / 140W |      1MiB / 16376MiB |      0%      Default |

|                               |                      |                  N/A |

+-------------------------------+----------------------+----------------------+

                                                                            

In [10]:
TARGET_MODEL = "mistralai/Mistral-7B-v0.1"

DEBUG = False

In [6]:
from peft import get_peft_config, PeftModel, PeftConfig, get_peft_model, LoraConfig, TaskType # type: ignore
from transformers import BitsAndBytesConfig
import torch
import numpy  as np
import random
import os

In [7]:
def set_seed(seed=42):
    '''Sets the seed of the entire notebook so results are the same every time we run.
    This is for REPRODUCIBILITY.'''
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ['PYTHONHASHSEED'] = str(seed)
    
set_seed(42)

In [8]:
import pandas as pd
train = pd.read_csv('input/train.csv')
train['Response'] = train['Response'].fillna('NA')

In [9]:
rows = []
for i, row in train.iterrows():
    question = row.Question
    response = row.Response
    context = train[(train.Question==question)&(train.Response!=response)].Response.values
    context = ' [SEP] '.join(context)
    train.loc[i, 'context'] = context

In [10]:
from sklearn.model_selection import GroupKFold
folds = GroupKFold(n_splits=5)
train['fold'] = -1
for i,(train_index, test_index) in enumerate(folds.split(train,train['target'], groups=train['Question'])): 
    train.loc[test_index,'fold'] = i

In [11]:
train['input'] = "Question: " + train['Question'] + '; Answer: ' + train['Response'] #+ '; Context: ' + train["context"]

In [12]:
train = train.rename(columns={'target': 'label'})

In [13]:
# load model with 4bit bnb

from peft import get_peft_config, PeftModel, PeftConfig, get_peft_model, LoraConfig, TaskType # type: ignore
from transformers import BitsAndBytesConfig
import torch


In [11]:
from transformers import AutoTokenizer, LlamaForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

In [12]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        
        all_param += param.numel()
        if param.requires_grad:
            print(_)
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [13]:
def preprocess_function(examples, max_length=512):
    return tokenizer(examples["input"], truncation=True, max_length=max_length, padding=True)

In [14]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

In [18]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy_val = accuracy_score(labels, predictions)
    
    return {
        "accuracy": accuracy_val,
    }

In [19]:
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / e_x.sum(axis=1, keepdims=True)

In [ ]:
from transformers import TrainingArguments, Trainer
from datasets import Dataset

steps = 5 if DEBUG else 50
for fold in range(5):
    print(f'Fold {fold+1}/5')

    valid_df = train[train["fold"] == fold]
    train_df = train[train["fold"] != fold]
    
    # from pandas
    train_ds = Dataset.from_pandas(train_df)
    valid_ds = Dataset.from_pandas(valid_df)
    
    train_tokenized_ds = train_ds.map(preprocess_function, batched=True)
    valid_tokenized_ds = valid_ds.map(preprocess_function, batched=True)
    
    # Model
    
    peft_config = LoraConfig(
        r=4,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        target_modules=[
            "q_proj",
            "v_proj"
        ],
    )

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    base_model = LlamaForSequenceClassification.from_pretrained(
        TARGET_MODEL,
        num_labels=7,
        quantization_config=bnb_config,
        device_map={"":0}
    )
    base_model.config.pretraining_tp = 1 # 1 is 7b
    base_model.config.pad_token_id = tokenizer.pad_token_id
    
    model = get_peft_model(base_model, peft_config)
    
    training_args = TrainingArguments(
        output_dir=f"outputs/fold{fold}",
        learning_rate=1e-4,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        max_grad_norm= 3,#0.3,
        optim='paged_adamw_32bit',
        lr_scheduler_type="cosine",
        num_train_epochs=3,
        weight_decay=0.0001,
        evaluation_strategy="steps",
        save_strategy="steps",
        save_steps = 100,
        eval_steps = 100,
        logging_steps= 100,
        load_best_model_at_end=True,
        push_to_hub=False,
        warmup_steps=10,
        report_to='none' # if DEBUG else 'wandb',
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized_ds,
        eval_dataset=valid_tokenized_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
        #compute_metrics=compute_metrics,
    )
    
    trainer.train()
    # validation 
    pred_output = trainer.predict(valid_tokenized_ds)
    logits = pred_output.predictions
    probas = softmax(logits)
    np.save(f'fold{fold}.npy', probas)

In [ ]:
# 0.976562 (step800), 0.9096(1100), 0.9379(1100), 0.9677 (800), 0.9838 (700)

In [ ]:
del trainer, model, base_model

In [20]:
# cuda cache clear
import torch
torch.cuda.empty_cache()

## Get OOF

In [15]:
from transformers import TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd
test = pd.read_csv('input/test.csv')
test['Response'] = test['Response'].fillna('NA')

rows = []
for i, row in test.iterrows():
    question = row.Question
    response = row.Response
    context = test[(test.Question==question)&(test.Response!=response)].Response.values
    context = ' [SEP] '.join(context)
    test.loc[i, 'context'] = context

test['input'] = "Question: " + test['Question'] + '; Answer: ' + test['Response']# + '; Context: ' + test["context"]

test_ds = Dataset.from_pandas(test)
test_tokenized_ds = test_ds.map(preprocess_function, batched=True)

  0%|          | 0/2 [00:00<?, ?ba/s]

In [16]:
target_cols = [f'target_{i}' for i in range(7)]

In [26]:
CKPTS =  ["checkpoint-800", "checkpoint-1100", "checkpoint-1100", "checkpoint-800", "checkpoint-700"]
final_preds = []
for fold, ckpt in enumerate(CKPTS):
    print("-------------------------------------------------------")
    print(f"Fold {fold}")
    print("-------------------------------------------------------")
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    base_model = LlamaForSequenceClassification.from_pretrained(
        TARGET_MODEL,
        num_labels=7,
        quantization_config=bnb_config,
        device_map={"":0}
    )
    base_model.config.pretraining_tp = 1 # 1 is 7b
    base_model.config.pad_token_id = tokenizer.pad_token_id
    
    model = PeftModel.from_pretrained(base_model, f"outputs/fold{fold}/{ckpt}")
    
    trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
)
    # valid
    valid_df = train[train["fold"] == fold]
    idxs = valid_df.index
    valid_ds = Dataset.from_pandas(valid_df)
    valid_tokenized_ds = valid_ds.map(preprocess_function, batched=True)
    
    pred_output = trainer.predict(valid_tokenized_ds)
    logits = pred_output.predictions
    probas = softmax(logits)
    train.loc[idxs, target_cols] = probas
    
    # test
    
    pred_output = trainer.predict(test_tokenized_ds)
    logits = pred_output.predictions
    probas = softmax(logits)
    final_preds.append(probas)
    

You are using a model of type mistral to instantiate a model of type llama. This is not supported for all configurations of models and can yield errors.


-------------------------------------------------------

Fold 0

-------------------------------------------------------


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at mistralai/Mistral-7B-v0.1 and are newly initialized: ['score.weight']

You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/1 [00:00<?, ?ba/s]

...

-------------------------------------------------------

Fold 4

-------------------------------------------------------


You are using a model of type mistral to instantiate a model of type llama. This is not supported for all configurations of models and can yield errors.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at mistralai/Mistral-7B-v0.1 and are newly initialized: ['score.weight']

You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/1 [00:00<?, ?ba/s]

In [34]:
test[target_cols] = np.mean(final_preds, 0)

In [36]:
test[['id'] + target_cols].to_csv('sub_LoRa_mistral_5folds.csv', index=False)

In [29]:
train.to_csv('train_oof_mistral.csv', index=False)

In [ ]:
## load

In [4]:
target_cols = [f'target_{i}' for i in range(7)]

In [17]:
from sklearn.metrics import log_loss
import pandas as pd
train = pd.read_csv('train_oof_mistral.csv')
log_loss(train['label'], train[target_cols])

0.9592193986280156

In [18]:
test[target_cols] = pd.read_csv('sub_LoRa_mistral_5folds.csv')[target_cols]